# Task 10: Risk Parity and Maximum Sharpe Ratio

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

cov = pd.read_csv('test5_2.csv').values
n = cov.shape[0]
print("Covariance matrix shape:", cov.shape)
print(cov)

Covariance matrix shape: (5, 5)
[[0.08497905 0.11678108 0.04230441 0.00898354 0.00387595]
 [0.11678108 0.16048451 0.05813615 0.01234549 0.00532646]
 [0.04230441 0.05813615 0.03744009 0.00596294 0.00257271]
 [0.00898354 0.01234549 0.00596294 0.00168834 0.00054633]
 [0.00387595 0.00532646 0.00257271 0.00054633 0.00031428]]


In [2]:
# --- 10.1: Risk Parity, Normal Assumption ---
def risk_parity_log(cov, budget):
    """Solve risk budgeting via log-barrier; normalize result to sum to 1."""
    n = len(budget)
    budget = np.array(budget, dtype=float)

    def objective(w):
        return 0.5 * w @ cov @ w - budget @ np.log(w)

    def gradient(w):
        return cov @ w - budget / w

    x0 = np.ones(n) / n
    bounds = [(1e-8, None)] * n
    res = minimize(objective, x0, jac=gradient, method='L-BFGS-B',
                   bounds=bounds, options={'ftol': 1e-20, 'gtol': 1e-12, 'maxiter': 5000})
    w = res.x / res.x.sum()
    return w

# Equal risk budgets
budget_101 = np.ones(n)
w_101 = risk_parity_log(cov, budget_101)
print("10.1 Risk Parity weights:", w_101)
print("Sum:", w_101.sum())

# Verify risk contributions are equal
rc = w_101 * (cov @ w_101)
print("Risk contributions:", rc / rc.sum())

out_101 = pd.DataFrame({'W': w_101})
out_101.to_csv('testout_10.1.csv', index=False)
print("\nSaved testout_10.1.csv")

10.1 Risk Parity weights: [0.03546374 0.02580619 0.0564694  0.2659201  0.61634058]
Sum: 1.0
Risk contributions: [0.2 0.2 0.2 0.2 0.2]

Saved testout_10.1.csv


In [3]:
# --- 10.2: Risk Parity, X5 gets 1/2 the risk budget of others ---
budget_102 = np.array([1.0, 1.0, 1.0, 1.0, 0.5])
w_102 = risk_parity_log(cov, budget_102)
print("10.2 Risk Parity (X5 half budget) weights:", w_102)
print("Sum:", w_102.sum())

# Verify risk contributions are proportional to budget
rc = w_102 * (cov @ w_102)
print("RC proportions:", rc / rc.sum(), "  expected:", budget_102 / budget_102.sum())

out_102 = pd.DataFrame({'W': w_102})
out_102.to_csv('testout_10.2.csv', index=False)
print("\nSaved testout_10.2.csv")

10.2 Risk Parity (X5 half budget) weights: [0.05024227 0.03656021 0.08039916 0.37860772 0.45419064]
Sum: 1.0
RC proportions: [0.22222222 0.22222222 0.22222222 0.22222222 0.11111111]   expected: [0.22222222 0.22222222 0.22222222 0.22222222 0.11111111]

Saved testout_10.2.csv


In [4]:
# --- 10.3: Maximum Sharpe Ratio, normal assumption, w >= 0 ---
means = pd.read_csv('test10_3_means.csv').values.flatten()
rf = 0.04
excess_ret = means - rf
print("Means:", means)
print("Excess returns:", excess_ret)

def neg_sharpe(w, cov, excess_ret):
    port_ret = w @ excess_ret
    port_vol = np.sqrt(w @ cov @ w)
    return -port_ret / port_vol

constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
bounds_103 = [(0.0, 1.0)] * n

best_result = None
# Try multiple starting points
for seed in range(20):
    rng = np.random.default_rng(seed)
    x0 = rng.dirichlet(np.ones(n))
    res = minimize(neg_sharpe, x0, args=(cov, excess_ret), method='SLSQP',
                   bounds=bounds_103, constraints=constraints,
                   options={'ftol': 1e-12, 'maxiter': 2000})
    if best_result is None or res.fun < best_result.fun:
        best_result = res

w_103 = best_result.x
# Zero out numerically negligible weights
w_103[w_103 < 1e-6] = 0.0

print("\n10.3 Max Sharpe (w>=0) weights:", w_103)
print("Sum:", w_103.sum())
sharpe_103 = (w_103 @ excess_ret) / np.sqrt(w_103 @ cov @ w_103)
print("Sharpe ratio:", sharpe_103)

out_103 = pd.DataFrame({'W': w_103})
out_103.to_csv('testout_10.3.csv', index=False)
print("\nSaved testout_10.3.csv")

Means: [0.09 0.08 0.07 0.06 0.05]
Excess returns: [0.05 0.04 0.03 0.02 0.01]

10.3 Max Sharpe (w>=0) weights: [0.         0.         0.         0.12131082 0.87868918]
Sum: 1.0000000000000415
Sharpe ratio: 0.5722360226382855

Saved testout_10.3.csv


In [5]:
# --- 10.4: Maximum Sharpe Ratio, normal assumption, 0.1 <= w <= 0.5 ---
bounds_104 = [(0.1, 0.5)] * n

best_result = None
for seed in range(20):
    rng = np.random.default_rng(seed)
    # Start from a feasible interior point
    x0 = np.ones(n) / n  # 0.2 each — within [0.1, 0.5]
    # Perturb slightly while staying feasible
    noise = rng.uniform(-0.05, 0.05, n)
    x0 = np.clip(x0 + noise, 0.1, 0.5)
    x0 = x0 / x0.sum()  # re-normalize (may briefly violate bounds, SLSQP handles it)
    res = minimize(neg_sharpe, x0, args=(cov, excess_ret), method='SLSQP',
                   bounds=bounds_104, constraints=constraints,
                   options={'ftol': 1e-12, 'maxiter': 2000})
    if best_result is None or res.fun < best_result.fun:
        best_result = res

w_104 = best_result.x

print("10.4 Max Sharpe (0.1<=w<=0.5) weights:", w_104)
print("Sum:", w_104.sum())
sharpe_104 = (w_104 @ excess_ret) / np.sqrt(w_104 @ cov @ w_104)
print("Sharpe ratio:", sharpe_104)

out_104 = pd.DataFrame({'W': w_104})
out_104.to_csv('testout_10.4.csv', index=False)
print("\nSaved testout_10.4.csv")

10.4 Max Sharpe (0.1<=w<=0.5) weights: [0.1 0.1 0.1 0.5 0.2]
Sum: 1.0000000000003673
Sharpe ratio: 0.22969300501941997

Saved testout_10.4.csv
